# Instructed pilot copying diagnostic
Attach the private instructed 78-pair Dataset and **training version 9** output through **Add Input**. Enable `HF_TOKEN`, select T4, and allow internet access. This notebook performs inference only: two seen training pairs (`p001`, `p014`) and two untouched test pairs (`p003`, `p045`). It compares base and adapted greedy outputs, tries one sampled adapted output per pair, and audits all 78 tokenized prompt/target lengths. It saves `copy_diagnostic.jsonl` and `copy_token_audit.json` under `/kaggle/working`. It does not retrain or modify the adapter.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0] or "0"


In [ ]:
%pip install -q -U "transformers>=4.50,<6" "peft>=0.15" "accelerate>=1.0" "bitsandbytes>=0.46"


In [ ]:
"""Frozen prompt formatting used by historical experiments."""

REWRITE_INSTRUCTION = (
    "Rewrite the passage below as formal academic economics prose. Preserve every "
    "substantive claim and qualification. Do not add facts, examples, or commentary. "
    "Return only the rewritten passage, without Markdown."
)

def generation_prompt(plain_english, instruction=False):
    if not instruction:
        return plain_english
    return f"{REWRITE_INSTRUCTION}\n\nPassage:\n{plain_english}"


"""Generate a side-by-side base/QLoRA comparison on untouched pilot test rows."""

import argparse
import json
import os
import re
from pathlib import Path



MODEL_ID = "google/gemma-3-1b-it"




def validate_inputs(dataset_path, adapter_dir):
    with Path(dataset_path).open(encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    if len(rows) != 78 or {name: sum(row.get("split") == name for row in rows)
                            for name in ("train", "validation", "test")} != {
                                "train": 54, "validation": 12, "test": 12}:
        raise ValueError("Expected the cleaned 78-pair pilot dataset with 54/12/12 splits")
    test_rows = [row for row in rows if row["split"] == "test"]
    if any(not isinstance(row.get("prompt"), str) or not isinstance(row.get("completion"), str)
           or not row["prompt"].strip() or not row["completion"].strip()
           for row in test_rows):
        raise ValueError("Test rows need nonempty prompt and completion fields")
    adapter_dir = Path(adapter_dir)
    config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
    if config.get("base_model_name_or_path") != MODEL_ID:
        raise ValueError(f"Adapter must be trained from {MODEL_ID}")
    if not (adapter_dir / "adapter_model.safetensors").is_file():
        raise FileNotFoundError("Missing adapter_model.safetensors")
    if any(row.get("prompt_format") == "rewrite_v1" for row in rows):
        evaluation_path = adapter_dir / "evaluation.json"
        if not evaluation_path.is_file() or json.loads(evaluation_path.read_text(encoding="utf-8")).get("prompt_format") != "rewrite_v1":
            raise ValueError("Attach the adapter from the instructed pilot, not the version 7 bare-prompt pilot")
    return test_rows


def write_claim_review_template(results, path):
    """Provide a sentence-level worksheet; factual judgments remain manual."""
    lines = [
        "# Meaning and style review",
        "",
        "For each source sentence, mark whether each output preserves, changes, or omits its claim."
        " Then record any added claims. Sentences are review aids, not automatic claim labels.",
        "",
    ]
    for result in results:
        source = result.get("plain_english", result["prompt"])
        sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", source.strip())
        lines.extend([f"## {', '.join(result['source_ids'])}", "", "### Source claims", ""])
        for index, sentence in enumerate(sentences, 1):
            lines.extend([
                f"{index}. {sentence}",
                "   - Base: [ ] preserved  [ ] changed  [ ] omitted",
                "   - Fine-tuned: [ ] preserved  [ ] changed  [ ] omitted",
            ])
        lines.extend([
            "", "### Unsupported additions", "", "- Base: ", "- Fine-tuned: ",
            "", "### Style and clarity notes", "", "- Base: ", "- Fine-tuned: ",
            "", "### Outputs", "", "**Base**", "", result["base_output"],
            "", "**Fine-tuned**", "", result["fine_tuned_output"], "",
            "**Reference**", "", result["reference"], "",
        ])
    Path(path).write_text("\n".join(lines), encoding="utf-8")


def load_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = None
    if token:
        os.environ["HF_TOKEN"] = token
    return token


def run_comparison(test_rows, adapter_dir, output_path, max_new_tokens=512,
                   instruction=False):
    os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0] or "0"
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError("A Kaggle GPU is required for model generation")
    if not load_hf_token():
        raise RuntimeError("Enable the Kaggle Secret named HF_TOKEN")

    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map={"": 0},
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        ),
    )
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    model.eval()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    def generate(input_ids):
        with torch.inference_mode():
            tokens = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        return tokenizer.decode(tokens[0, input_ids.shape[-1]:], skip_special_tokens=True).strip()

    results = []
    with output_path.open("w", encoding="utf-8") as handle:
        for index, row in enumerate(test_rows, 1):
            # Both models get identical tokens; instructed mode probes task ambiguity.
            prompt = generation_prompt(row.get("plain_english", row["prompt"]), instruction) if instruction else row["prompt"]
            encoded = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
            input_ids = encoded["input_ids"].to("cuda:0")
            with model.disable_adapter():
                base_output = generate(input_ids)
            fine_tuned_output = generate(input_ids)
            result = {
                "source_ids": row["source_ids"],
                "prompt": row["prompt"],
                "plain_english": row.get("plain_english", row["prompt"]),
                "generation_prompt": prompt,
                "reference": row["completion"],
                "base_output": base_output,
                "fine_tuned_output": fine_tuned_output,
            }
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")
            handle.flush()
            results.append(result)
            print(f"Generated comparison {index}/{len(test_rows)}")
    print(f"Saved comparisons to {output_path}")
    review_path = output_path.with_name(output_path.stem + "_claim_review.md")
    write_claim_review_template(results, review_path)
    print(f"Saved claim-review worksheet to {review_path}")


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--adapter", required=True)
    parser.add_argument("--output", default="/kaggle/working/fama_style_comparison.jsonl")
    parser.add_argument("--max-new-tokens", type=int, default=512)
    parser.add_argument("--source-ids", help="comma-separated IDs of untouched test examples")
    parser.add_argument("--instruction", action="store_true",
                        help="prepend an explicit rewrite instruction for both models")
    parser.add_argument("--require-rewrite", action="store_true",
                        help="require that every dataset prompt already has the rewrite instruction")
    parser.add_argument("--check", action="store_true")
    args = parser.parse_args(argv)
    if args.max_new_tokens <= 0:
        raise ValueError("max_new_tokens must be positive")
    test_rows = validate_inputs(args.dataset, args.adapter)
    if args.require_rewrite and any(
        row.get("prompt_format") != "rewrite_v1" or
        row["prompt"] != generation_prompt(row.get("plain_english", ""), instruction=True)
        for row in test_rows
    ):
        raise ValueError("Test prompts must use the instructed rewrite_v1 format")
    if args.source_ids:
        selected = {part.strip() for part in args.source_ids.split(",") if part.strip()}
        test_rows = [row for row in test_rows if selected.intersection(row["source_ids"])]
        if {row_id for row in test_rows for row_id in row["source_ids"]} != selected:
            raise ValueError("source_ids must identify untouched test examples")
    print(f"Validated {len(test_rows)} untouched test pairs and Gemma adapter")
    if args.check:
        print("Dry-run OK. No model downloaded or generated.")
        return
    run_comparison(test_rows, args.adapter, args.output, args.max_new_tokens,
                   args.instruction)



"""Small, inference-only check of copying in the instructed Gemma pilot."""

import argparse
import difflib
import json
import os
from pathlib import Path



EXAMPLE_IDS = ("p001", "p014", "p003", "p045")
EXPECTED_SPLITS = ("train", "train", "test", "test")
MAX_LENGTH = 1024


def select_examples(dataset_path, adapter_dir):
    with Path(dataset_path).open(encoding="utf-8") as handle:
        rows = [json.loads(line) for line in handle if line.strip()]
    counts = {split: sum(row.get("split") == split for row in rows)
              for split in ("train", "validation", "test")}
    if len(rows) != 78 or counts != {"train": 54, "validation": 12, "test": 12}:
        raise ValueError("Expected the instructed 78-pair dataset with 54/12/12 splits")
    if any(row.get("prompt_format") != "rewrite_v1" or
           row.get("prompt") != generation_prompt(row.get("plain_english", ""), instruction=True) or
           not isinstance(row.get("completion"), str) or not row["completion"].strip()
           for row in rows):
        raise ValueError("Dataset prompts and completions must use rewrite_v1")
    selected = []
    for source_id, split in zip(EXAMPLE_IDS, EXPECTED_SPLITS):
        matches = [row for row in rows if row.get("source_ids") == [source_id]
                   and row.get("split") == split]
        if len(matches) != 1:
            raise ValueError(f"Expected one {split} row with source ID {source_id}")
        selected.append(matches[0])
    adapter_dir = Path(adapter_dir)
    config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
    evaluation = json.loads((adapter_dir / "evaluation.json").read_text(encoding="utf-8"))
    if config.get("base_model_name_or_path") != MODEL_ID or evaluation.get("model_id") != MODEL_ID:
        raise ValueError(f"Adapter must be trained from {MODEL_ID}")
    if evaluation.get("prompt_format") != "rewrite_v1":
        raise ValueError("Attach the version 9 instructed adapter")
    if not (adapter_dir / "adapter_model.safetensors").is_file():
        raise FileNotFoundError("Missing adapter_model.safetensors")
    return rows, selected


def token_audit(rows, tokenizer):
    audit = []
    for row in rows:
        user = [{"role": "user", "content": row["prompt"]}]
        conversation = user + [{"role": "assistant", "content": row["completion"]}]
        prompt_ids = tokenizer.apply_chat_template(
            user, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
        )["input_ids"][0].tolist()
        full_ids = tokenizer.apply_chat_template(
            conversation, tokenize=True, return_dict=True, return_tensors="pt",
        )["input_ids"][0].tolist()
        prompt_rendered = tokenizer.apply_chat_template(user, tokenize=False,
                                                        add_generation_prompt=True)
        rendered = tokenizer.apply_chat_template(conversation, tokenize=False)
        prompt_is_prefix = full_ids[:len(prompt_ids)] == prompt_ids
        audit.append({
            "source_ids": row["source_ids"],
            "split": row["split"],
            "prompt_tokens": len(prompt_ids),
            "full_tokens": len(full_ids),
            "prompt_text_is_prefix": rendered.startswith(prompt_rendered),
            "prompt_is_prefix": prompt_is_prefix,
            "completion_tokens": len(full_ids) - len(prompt_ids) if prompt_is_prefix else None,
            "over_1024_tokens": len(full_ids) > MAX_LENGTH,
            "target_in_rendered_assistant": row["completion"] in rendered,
        })
    return audit


def copy_stats(output, source):
    return {
        "exact_copy": output == source,
        "similarity_to_input": round(difflib.SequenceMatcher(None, output, source).ratio(), 4),
    }


def run_diagnostic(rows, selected, adapter_dir, output_dir):
    os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0] or "0"
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError("A Kaggle GPU is required for inference")
    if not load_hf_token():
        raise RuntimeError("Enable the Kaggle Secret named HF_TOKEN")

    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
    audit = token_audit(rows, tokenizer)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "copy_token_audit.json").write_text(
        json.dumps({"max_length": MAX_LENGTH, "rows": audit}, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"Token audit saved: {len(audit)} rows; "
          f"{sum(item['over_1024_tokens'] for item in audit)} over {MAX_LENGTH} tokens; "
          f"{sum(not item['prompt_text_is_prefix'] for item in audit)} text-prefix mismatches; "
          f"{sum(not item['prompt_is_prefix'] for item in audit)} token-prefix mismatches; "
          f"{sum(not item['target_in_rendered_assistant'] for item in audit)} target mismatches")
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map={"": 0},
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        ),
    )
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    model.eval()

    def generate(encoded, *, sample=False):
        options = {"max_new_tokens": 256, "do_sample": sample,
                   "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id}
        if sample:
            options.update(temperature=0.7, top_p=0.9)
        with torch.inference_mode():
            tokens = model.generate(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
                **options,
            )
        return tokenizer.decode(tokens[0, encoded["input_ids"].shape[-1]:],
                                skip_special_tokens=True).strip()

    results = []
    for index, row in enumerate(selected, 1):
        encoded = tokenizer.apply_chat_template(
            [{"role": "user", "content": row["prompt"]}],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        encoded = {key: value.to("cuda:0") for key, value in encoded.items()}
        with model.disable_adapter():
            base_greedy = generate(encoded)
        adapted_greedy = generate(encoded)
        torch.manual_seed(42 + index)
        adapted_sampled = generate(encoded, sample=True)
        results.append({
            "source_ids": row["source_ids"],
            "split": row["split"],
            "plain_english": row["plain_english"],
            "reference": row["completion"],
            "base_greedy": base_greedy,
            "adapted_greedy": adapted_greedy,
            "adapted_sampled": adapted_sampled,
            "base_greedy_copy": copy_stats(base_greedy, row["plain_english"]),
            "adapted_greedy_copy": copy_stats(adapted_greedy, row["plain_english"]),
            "adapted_sampled_copy": copy_stats(adapted_sampled, row["plain_english"]),
        })
        print(f"Generated diagnostic {index}/{len(selected)}: {row['split']} {row['source_ids'][0]}")

    with (output_dir / "copy_diagnostic.jsonl").open("w", encoding="utf-8") as handle:
        for result in results:
            handle.write(json.dumps(result, ensure_ascii=False) + "\n")
    print(f"Saved {len(results)} comparisons and {len(audit)} token audits to {output_dir}")


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--adapter", required=True)
    parser.add_argument("--output-dir", default="/kaggle/working")
    parser.add_argument("--check", action="store_true", help="validate inputs without a GPU or model download")
    args = parser.parse_args(argv)
    rows, selected = select_examples(args.dataset, args.adapter)
    print(f"Validated {len(rows)} pilot rows and {len(selected)} diagnostic examples")
    if args.check:
        print("Dry-run OK. No model downloaded or generated.")
        return
    run_diagnostic(rows, selected, args.adapter, args.output_dir)



datasets = list(Path("/kaggle/input").rglob("fama_jensen_1983_pilot_instructed.jsonl"))
adapters = list(Path("/kaggle/input").rglob("gemma3_adapter/adapter_config.json"))
if len(datasets) != 1 or len(adapters) != 1:
    raise FileNotFoundError(f"Expected one instructed dataset and one version 9 adapter input; found {len(datasets)} datasets and {len(adapters)} adapters")
main(["--dataset", str(datasets[0]), "--adapter", str(adapters[0].parent), "--output-dir", "/kaggle/working"])
